In [0]:
import sys
import os

# Add the parent directory to Python path to enable imports from utilities
parent_dir = os.path.dirname(os.path.abspath(os.getcwd()))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from utilities.validation_utilities import validate_streaming_table,validate_mv,validate_delta

In [0]:
dbutils.widgets.text("unity_catalog","learn_adb_fikrat")
uc_name=dbutils.widgets.get("unity_catalog")
spark.sql(f'USE CATALOG {uc_name}');

In [0]:
display(dbutils.fs.ls("dbfs:/databricks-datasets/retail-org/"))

In [0]:
display(spark.read.format("json").load("dbfs:/databricks-datasets/retail-org/sales_orders/"))

In [0]:
display(spark.read.csv("dbfs:/databricks-datasets/retail-org/customers/"))

## Streaming Tables

**Step 1.** Add a streaming table for sales orders. 
- The streaming table will source data from this path: dbfs:/databricks-datasets/retail-org/sales_orders/ 
- Name the target table as 'sales' and place it in pipeline's default catalog's _Bronze_ schema

In [0]:
validate_streaming_table('bronze','sales')

**Step 2.** Create a streaming table for customers. 
- The streaming table will source data from this path: dbfs:/databricks-datasets/retail-org/customers/ 
- Name the target table as 'customers' and place it in pipeline's default catalog's _Bronze_ schema

In [0]:
validate_streaming_table('bronze','customers')

**Step 3**. Create a streaming table named 'sales_orders' in the Silver schema, sourced from _bronze.sales_ table. **Instructions**: 
- The table sales_orders should include only the following columns: order_number,order_datetime,customer_id, 
 customer_name,number_of_line_items

In [0]:
validate_streaming_table('silver','sales_orders')

## Data Quality Controls

**Step 4.** Create a streaming table named 'customers' in Silver schema, sourced from _bronze.customers_ table. Instructions: 
- Use Data Quality validation expectations to filter out  customers having empty city attributes.
- Use Data Quality validation expectations of warning type for customers having empty region attributes.
- The target table should include only the following columns: customer_id, customer_name, city, state,region

In [0]:
validate_streaming_table('silver','customers')

## Materialized views

**Step 5**. Create a materialized view named 'mv_sales_order_items' in the Silver schema, sourced from _Bronze.sales_ table.The definion of materilazied view should include following transformations:
- Add Identity column named item_id, using _monotonically_increasing_id_() function. 
- Parse ordered_products column to separate each ordered item, using _explode_ function.
- Extract sub-fields under ordered_products
- Calculate amount field by multiplying _qty_ to _price_ column
- The table should include only the following columns: "item_id","order_number","product_id", "product_name", "currency","unit","price", "qty","amount"


In [0]:
validate_mv('silver','mv_sales_order_items')

**Step 6.** Create a materialized view named 'mv_sales_order_aggregates' in the Silver schema. **Instructions**:
- Join silver.sales_orders and silver.mv_sales_order_items tables on order_number column.
- Group by order_number, calculate the sum of amount column, and name it total_amount
- The target table should be a Delta Lake table and should include only the following columns: "order_number","total_amount"

In [0]:
validate_mv('silver','mv_sales_order_aggregates')

**Step 7**.Create a materialized view named 'mv_sales_orders_customers' in the Silver schema. **Instructions**:
- Join _silver.mv_sales_order_aggregates, silver.sales_orders and silver.customers_ tables on order_number column.
- Group by _order_number_, calculate the sum of _amount_ column, and name it _total_amount_
- The target table should be a Delta Lake table and should include only the following columns: "order_number","order_datetime","number_of_line_items","total_amount","customer_id","customer_name","city","state"

In [0]:
validate_mv('silver','mv_sales_orders_customers')

**Step 8**. TO DO: Create a materialized view named '_mv_sales_orders_aggregates_by_state_' in the Silver schema. **Instructions**:
- Group by "state","city" columns and calculate the sum of total_amount column, and name it as _total_amount_$_
- The target table should be a Delta Lake table and should include only the following columns: "state","city","total_amount_$"
- Order by "state","city" columns

In [0]:
validate_mv('silver','mv_sales_orders_aggregates_by_state')

**Step 9**. Create a Delta Lake sink named '_sales_orders_aggregates_delta_lake_' in the _Silver_ schema. The target table should be a Delta Lake table and should include only the following columns: "state","city","total_amount_$"
- Tip: Use catalog name paramater to specify the catalog name for the target table

In [0]:
validate_delta('silver','sales_orders_aggregates_delta_lake')

In [0]:
%sql
use bronze

In [0]:
%sql
select table_type from INFORMATION_SCHEMA.TABLES
                 where table_name= 'sales_orders_aggregates_delta_lake' and table_schema='silver'

In [0]:
df=spark.read.load("/databricks-datasets/tpch/delta-001/orders")
display(df)

In [0]:
display(dbutils.fs.ls("/databricks-datasets/tpch/delta-001/"))

In [0]:
df=spark.read.load("/databricks-datasets/tpch/delta-001/lineitem")
display(df)